In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

# הזנה קדמית קלאסית ובקרת זרימה (מעגלים דינמיים)

<Accordion>
<AccordionItem title="Package versions">

הקוד בדף זה פותח תוך שימוש בדרישות הבאות.
אנחנו ממליצים להשתמש בגרסאות אלו או בגרסאות חדשות יותר.

```
qiskit[all]~=2.4.0
```
</AccordionItem>
</Accordion>
מעגלים דינמיים הם כלים רבי עוצמה שבהם אפשר למדוד qubits באמצע ביצוע מעגל קוונטי, ואז לבצע פעולות לוגיקה קלאסית בתוך המעגל, בהתבסס על תוצאות אותן מדידות אמצע-מעגל. תהליך זה ידוע גם בשם _הזנה קדמית קלאסית_. למרות שאלו עדיין ימיה הראשונים של ההבנה כיצד לנצל בצורה הטובה ביותר את המעגלים הדינמיים, קהילת המחקר הקוונטי כבר זיהתה מספר מקרי שימוש, כגון הבאים:

* הכנת מצב קוונטי יעילה, כגון [מצב GHZ](https://journals.aps.org/prxquantum/abstract/10.1103/PRXQuantum.5.030339), [מצב W](https://arxiv.org/abs/2403.07604) (למידע נוסף על מצב W, ראה גם ["State preparation by shallow circuits using feed forward"](https://arxiv.org/abs/2307.14840)), ומחלקה רחבה של [מצבי מכפלת מטריצה](https://arxiv.org/abs/2404.16083)
* [שזירה ארוכת טווח יעילה](https://journals.aps.org/prxquantum/abstract/10.1103/PRXQuantum.5.030339) בין qubits על אותו שבב באמצעות מעגלים רדודים
* [דגימה יעילה של מעגלים דמויי IQP](https://arxiv.org/pdf/2505.04705)

Qiskit תומך בארבעה מבני זרימת בקרה להזנה קדמית קלאסית, כל אחד מיושם כמתודה על [`QuantumCircuit`](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.QuantumCircuit). המבנים והמתודות המקבילות להם הם:

- משפט If - [`QuantumCircuit.if_test`](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.QuantumCircuit#if_test)
- משפט Switch - [`QuantumCircuit.switch`](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.QuantumCircuit#switch)
- לולאת For - [`QuantumCircuit.for_loop`](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.QuantumCircuit#for_loop)
- לולאת While - [`QuantumCircuit.while_loop`](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.QuantumCircuit#while_loop)

כל אחת מהמתודות הללו מחזירה [מנהל הקשר](https://docs.python.org/3/reference/datamodel.html#with-statement-context-managers) ומשמשת בדרך כלל במשפט `with`. שאר המדריך מסביר כל אחד ממבנים אלה וכיצד להשתמש בהם.

> **Caution:** ישנן כמה מגבלות של פעולות הזנה קדמית קלאסית וזרימת בקרה על חומרה קוונטית שעשויות להשפיע על התוכנית שלך. למידע נוסף, ראה [ביצוע מעגלים דינמיים](/guides/execute-dynamic-circuits).

## משפט `if`
משפט ה-`if` משמש לביצוע פעולות באופן מותנה בהתבסס על ערכו של סיבית קלאסי או רגיסטר.

בדוגמה הבאה, אנחנו מיישמים Gate הדמארד על qubit ומודדים אותו. אם התוצאה היא 1, אז אנחנו מיישמים Gate X על ה-Qubit, מה שהאפקט שלו הוא היפוכו בחזרה למצב 0. לאחר מכן אנחנו מודדים את ה-Qubit שוב. תוצאת המדידה המתקבלת צריכה להיות 0 עם הסתברות של 100%.

In [1]:
from qiskit.circuit import QuantumCircuit, QuantumRegister, ClassicalRegister

qubits = QuantumRegister(1)
clbits = ClassicalRegister(1)
circuit = QuantumCircuit(qubits, clbits)
(q0,) = qubits
(c0,) = clbits

circuit.h(q0)
circuit.measure(q0, c0)
with circuit.if_test((c0, 1)):
    circuit.x(q0)
circuit.measure(q0, c0)
circuit.draw("mpl")

# example output counts: {'0': 1024}

<Image src="../docs/images/guides/classical-feedforward-and-control-flow/extracted-outputs/101aaa8f-7061-4924-9b50-806d7e1ab728-0.avif" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/classical-feedforward-and-control-flow/extracted-outputs/101aaa8f-7061-4924-9b50-806d7e1ab728-0.avif)

ניתן לתת למשפט ה-`with` יעד השמה שהוא עצמו מנהל הקשר שניתן לאחסן ולהשתמש בו לאחר מכן ליצירת בלוק else, אשר מבוצע בכל פעם שתוכן בלוק ה-`if` *אינו* מבוצע.

בדוגמה הבאה, אנחנו מאתחלים רגיסטרים עם שני qubits ושני סיביות קלאסיות. אנחנו מיישמים Gate הדמארד על ה-Qubit הראשון ומודדים אותו. אם התוצאה היא 1, אז אנחנו מיישמים Gate הדמארד על ה-Qubit השני; אחרת, אנחנו מיישמים Gate X על ה-Qubit השני. לבסוף, אנחנו מודדים גם את ה-Qubit השני.

In [2]:
qubits = QuantumRegister(2)
clbits = ClassicalRegister(2)
circuit = QuantumCircuit(qubits, clbits)
(q0, q1) = qubits
(c0, c1) = clbits

circuit.h(q0)
circuit.measure(q0, c0)
with circuit.if_test((c0, 1)) as else_:
    circuit.h(q1)
with else_:
    circuit.x(q1)
circuit.measure(q1, c1)

circuit.draw("mpl")

# example output counts: {'01': 260, '11': 272, '10': 492}

<Image src="../docs/images/guides/classical-feedforward-and-control-flow/extracted-outputs/1f6737fe-bc45-4d0c-b7b4-1096e2d7e14a-0.avif" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/classical-feedforward-and-control-flow/extracted-outputs/1f6737fe-bc45-4d0c-b7b4-1096e2d7e14a-0.avif)

בנוסף לתנאי על סיבית קלאסי בודד, ניתן גם להתנות על ערכו של רגיסטר קלאסי המורכב ממספר סיביות.

בדוגמה הבאה, אנחנו מיישמים Gates הדמארד על שני qubits ומודדים אותם. אם התוצאה היא `01`, כלומר ה-Qubit הראשון הוא 1 וה-Qubit השני הוא 0, אז אנחנו מיישמים Gate X על qubit שלישי. לבסוף, אנחנו מודדים את ה-Qubit השלישי. שים לב שלשם הבהירות, בחרנו לציין את מצב הסיבית הקלאסית השלישית, שהוא 0, בתנאי ה-`if`. בציור המעגל, התנאי מסומן על ידי עיגולים על הסיביות הקלאסיות שמתנים עליהן. עיגול מלא מציין תנאי על 1, בעוד שעיגול עם מתאר מציין תנאי על 0.

In [3]:
qubits = QuantumRegister(3)
clbits = ClassicalRegister(3)
circuit = QuantumCircuit(qubits, clbits)
(q0, q1, q2) = qubits
(c0, c1, c2) = clbits

circuit.h([q0, q1])
circuit.measure(q0, c0)
circuit.measure(q1, c1)
with circuit.if_test((clbits, 0b001)):
    circuit.x(q2)
circuit.measure(q2, c2)

circuit.draw("mpl")

# example output counts: {'101': 269, '011': 260, '000': 252, '010': 243}

<Image src="../docs/images/guides/classical-feedforward-and-control-flow/extracted-outputs/37ec3fa6-04b5-4165-b8d2-bae5fd238331-0.avif" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/classical-feedforward-and-control-flow/extracted-outputs/37ec3fa6-04b5-4165-b8d2-bae5fd238331-0.avif)

## משפט Switch
משפט ה-switch משמש לבחירת פעולות בהתבסס על ערכו של סיבית קלאסי או רגיסטר. הוא דומה למשפט if, אך ניתן לציין מקרים נוספים עבור לוגיקת ההסתעפות. הדוגמה הבאה מיישמת Gate הדמארד על qubit ומודדת אותו. אם התוצאה היא 0, מיישמים Gate X על ה-Qubit, ואם התוצאה היא 1, מיישמים Gate Z. תוצאת המדידה המתקבלת צריכה להיות 1 עם הסתברות של 100%.

In [4]:
qubits = QuantumRegister(1)
clbits = ClassicalRegister(1)
circuit = QuantumCircuit(qubits, clbits)
(q0,) = qubits
(c0,) = clbits

circuit.h(q0)
circuit.measure(q0, c0)
with circuit.switch(c0) as case:
    with case(0):
        circuit.x(q0)
    with case(1):
        circuit.z(q0)
circuit.measure(q0, c0)

circuit.draw("mpl")

# example output counts: {'1': 1024}

<Image src="../docs/images/guides/classical-feedforward-and-control-flow/extracted-outputs/fc2bc3c3-eab1-41f0-b696-5e8b30155d55-0.avif" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/classical-feedforward-and-control-flow/extracted-outputs/fc2bc3c3-eab1-41f0-b696-5e8b30155d55-0.avif)

מכיוון שהדוגמה למעלה השתמשה בסיבית קלאסית בודדת, היו רק שני מקרים אפשריים, ולכן ניתן היה להשיג את אותה תוצאה באמצעות משפט if-else. משפט ה-switch שימושי בעיקר כאשר מסתעפים על ערכו של רגיסטר קלאסי המורכב ממספר סיביות. הדוגמה הבאה מראה כיצד לבנות מקרה ברירת מחדל, אשר מבוצע אם אף אחד מהמקרים הקודמים אינו תואם. שים לב שבמשפט switch, רק אחד מהבלוקים מבוצע. אין fallthrough.

הדוגמה הבאה מיישמת Gates הדמארד על שני qubits ומודדת אותם. אם התוצאה היא 00 או 11, מיישמים Gate Z על ה-Qubit השלישי. אם התוצאה היא 01, מיישמים Gate Y. אם אף אחד מהמקרים הקודמים לא תאם, מיישמים Gate X. לבסוף, מודדים את ה-Qubit השלישי.

In [5]:
qubits = QuantumRegister(3)
clbits = ClassicalRegister(3)
circuit = QuantumCircuit(qubits, clbits)
(q0, q1, q2) = qubits
(c0, c1, c2) = clbits

circuit.h([q0, q1])
circuit.measure(q0, c0)
circuit.measure(q1, c1)
with circuit.switch(clbits) as case:
    with case(0b000, 0b011):
        circuit.z(q2)
    with case(0b001):
        circuit.y(q2)
    with case(case.DEFAULT):
        circuit.x(q2)
circuit.measure(q2, c2)

circuit.draw("mpl")

# example output counts: {'101': 267, '110': 249, '011': 265, '000': 243}

<Image src="../docs/images/guides/classical-feedforward-and-control-flow/extracted-outputs/a5d43b4c-c538-4f34-8cf3-92c2c0d26fdd-0.avif" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/classical-feedforward-and-control-flow/extracted-outputs/a5d43b4c-c538-4f34-8cf3-92c2c0d26fdd-0.avif)

## לולאת For
לולאת for משמשת לאיטרציה על רצף של ערכים קלאסיים וביצוע פעולות מסוימות בכל איטרציה.

הדוגמה הבאה משתמשת בלולאת for להחלת 5 Gates X על qubit ולאחר מכן מודדת אותו. מכיוון שהיא מבצעת מספר אי-זוגי של Gates X, האפקט הכולל הוא להפוך את ה-Qubit ממצב 0 למצב 1.

In [6]:
qubits = QuantumRegister(1)
clbits = ClassicalRegister(1)
circuit = QuantumCircuit(qubits, clbits)
(q0,) = qubits
(c0,) = clbits

with circuit.for_loop(range(5)) as _:
    circuit.x(q0)
circuit.measure(q0, c0)

circuit.draw("mpl")

# example output counts: {'1': 1024}

<Image src="../docs/images/guides/classical-feedforward-and-control-flow/extracted-outputs/53a26ce5-3564-47a0-8803-c9c46db86923-0.avif" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/classical-feedforward-and-control-flow/extracted-outputs/53a26ce5-3564-47a0-8803-c9c46db86923-0.avif)

## לולאת While
לולאת while משמשת לחזרה על הוראות כל עוד תנאי מסוים מתקיים.

הדוגמה הבאה מיישמת Gates הדמארד על שני qubits ומודדת אותם. לאחר מכן היא יוצרת לולאת while שחוזרת על הליך זה כל עוד תוצאת המדידה היא 11. כתוצאה מכך, המדידה הסופית לעולם לא צריכה להיות 11, כאשר שאר האפשרויות מופיעות בתדירות שווה בקירוב.

In [7]:
qubits = QuantumRegister(2)
clbits = ClassicalRegister(2)
circuit = QuantumCircuit(qubits, clbits)

q0, q1 = qubits
c0, c1 = clbits

circuit.h([q0, q1])
circuit.measure(q0, c0)
circuit.measure(q1, c1)
with circuit.while_loop((clbits, 0b11)):
    circuit.h([q0, q1])
    circuit.measure(q0, c0)
    circuit.measure(q1, c1)

circuit.draw("mpl")

# example output counts: {'01': 334, '10': 368, '00': 322}

<Image src="../docs/images/guides/classical-feedforward-and-control-flow/extracted-outputs/174a9675-3c8b-4b5e-808e-f7e0f8b9c805-0.avif" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/classical-feedforward-and-control-flow/extracted-outputs/174a9675-3c8b-4b5e-808e-f7e0f8b9c805-0.avif)

## ביטויים קלאסיים
מודול הביטויים הקלאסיים של Qiskit [`qiskit.circuit.classical`](https://docs.quantum.ibm.com/api/qiskit/circuit_classical) מכיל ייצוג חקרני של פעולות זמן ריצה על ערכים קלאסיים במהלך ביצוע מעגל.

הדוגמה הבאה מראה שניתן להשתמש בחישוב הזוגיות ליצירת מצב GHZ בן n-Qubit באמצעות מעגלים דינמיים. ראשית, צור $n/2$ זוגות Bell על qubits סמוכים. לאחר מכן, חבר את הזוגות הללו יחד באמצעות שכבה של Gates CNOT בין הזוגות. לאחר מכן מודדים את ה-Qubit היעד של כל Gates CNOT הקודמים ומאפסים כל qubit מדוד למצב $\vert 0 \rangle$. מיישמים $X$ על כל אתר שלא נמדד שעבורו זוגיות כל הסיביות הקודמות היא אי-זוגית. לבסוף, Gates CNOT מוחלים על qubits המדודים כדי לשחזר את השזירה שאבדה במדידה.

בחישוב הזוגיות, האלמנט הראשון של הביטוי הנבנה כולל הרמת אובייקט Python `mr[0]` לצומת [`Value`](https://docs.quantum.ibm.com/api/qiskit/circuit_classical#value) (הפונקציה `lift` משמשת להפיכת אובייקטים שרירותיים לביטויים קלאסיים). אין צורך בכך עבור `mr[1]` ורגיסטרים הקלאסיים האפשריים הבאים, מכיוון שהם קלטים ל-`expr.bit_xor`, וכל הרמה נחוצה נעשית אוטומטית במקרים אלה. ניתן לבנות ביטויים כאלה בלולאות ובמבנים אחרים.

In [8]:
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister
from qiskit.circuit.classical import expr

num_qubits = 8
if num_qubits % 2 or num_qubits < 4:
    raise ValueError("num_qubits must be an even integer ≥ 4")
meas_qubits = list(range(2, num_qubits, 2))  # qubits to measure and reset

qr = QuantumRegister(num_qubits, "qr")
mr = ClassicalRegister(len(meas_qubits), "m")
qc = QuantumCircuit(qr, mr)

# Create local Bell pairs
qc.reset(qr)
qc.h(qr[::2])
for ctrl in range(0, num_qubits, 2):
    qc.cx(qr[ctrl], qr[ctrl + 1])

# Glue neighboring pairs
for ctrl in range(1, num_qubits - 1, 2):
    qc.cx(qr[ctrl], qr[ctrl + 1])

# Measure boundary qubits between pairs,reset to 0
for k, q in enumerate(meas_qubits):
    qc.measure(qr[q], mr[k])
    qc.reset(qr[q])

# Parity-conditioned X corrections
# Each non-measured qubit gets flipped iff the parity (XOR) of all
# preceding measurement bits is 1
for tgt in range(num_qubits):
    if tgt in meas_qubits:  # skip measured qubits
        continue
    # all measurement registers whose physical qubit index < tgt
    left_bits = [k for k, q in enumerate(meas_qubits) if q < tgt]
    if not left_bits:  # skip if list empty
        continue

    # build XOR-parity expression
    parity = expr.lift(
        mr[left_bits[0]]
    )  # lift the first bit to Value so it will be treated like a boolean.
    for k in left_bits[1:]:
        parity = expr.bit_xor(
            mr[k], parity
        )  # calculate parity with all other bits
    with qc.if_test(parity):  # Add X if parity is 1
        qc.x(qr[tgt])

# Re-entangle measured qubits
for ctrl in range(1, num_qubits - 1, 2):
    qc.cx(qr[ctrl], qr[ctrl + 1])

In [9]:
qc.draw(output="mpl", style="iqp", idle_wires=False, fold=-1)

<Image src="../docs/images/guides/classical-feedforward-and-control-flow/extracted-outputs/d2fdf38a-e874-4de1-9a79-08aab97f9ecc-0.avif" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/classical-feedforward-and-control-flow/extracted-outputs/d2fdf38a-e874-4de1-9a79-08aab97f9ecc-0.avif)

<span id="store"></span>
### `Store`

ניתן להשתמש בהוראת [`store`](https://docs.quantum.ibm.com/api/qiskit/circuit#store) כדי לשמור את תוצאת ביטוי קלאסי, אם ביטוי זה ישמש שוב ושוב. פעולות מקביליות אוטומטית, מה שהופך את הקוד ליעיל הרבה יותר בזמן ריצה.

לדוגמה, יותר טבעי ויעיל יותר בזמן ריצה לכתוב $B[0] \oplus B[1] \oplus B[2] \ldots$, כאשר $B = \neg A$, מאשר $(\neg A[0]) \oplus (\neg A[1]) \oplus (\neg A[2]) \ldots$.  הראשון מחשב את השלילה בשלב מקבילי יחיד לפני שרשרת ה-XOR, במקום לחשב כל שלילה ברצף בתוך הביטוי.

דוגמה מלאה:

In [10]:
from qiskit.circuit import QuantumCircuit, QuantumRegister, ClassicalRegister
from qiskit.circuit.classical import expr

qregs = QuantumRegister(4, "q")
creg = ClassicalRegister(3, "c")
# temp is a plain ClassicalRegister used as the store target
temp = ClassicalRegister(3, "temp")
qc = QuantumCircuit(qregs, creg, temp)

qc.h([0, 1, 2])
qc.measure([0, 1, 2], creg)

# Store bit-NOT of the full 3-bit register into temp
qc.store(temp, expr.bit_not(creg))

# Compute parity of temp using bit-indexed XOR
parity = expr.bit_xor(
    expr.bit_xor(expr.index(temp, 0), expr.index(temp, 1)),
    expr.index(temp, 2),
)

# Flip q3 if parity of ~creg is 1
with qc.if_test(parity):
    qc.x(3)

qc.measure([0, 1, 2], creg)

qc.draw("mpl")

<Image src="../docs/images/guides/classical-feedforward-and-control-flow/extracted-outputs/f76db731-afa1-4777-9482-25376aa86175-0.avif" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/classical-feedforward-and-control-flow/extracted-outputs/f76db731-afa1-4777-9482-25376aa86175-0.avif)

## הצעדים הבאים
> **Tip:** - למד כיצד ליישם פירוק דינמי מדויק באמצעות [stretch](/guides/stretch).
> - השתמש ב-[ויזואליזציית לוח הזמנים של מעגל](/guides/qiskit-runtime-circuit-timing) לניפוי שגיאות ואופטימיזציה של המעגלים הדינמיים שלך.
> - [הרץ מעגלים דינמיים](/guides/execute-dynamic-circuits).